# 保险客服真人级数字人视频生成
**零成本 · Google Colab 免费 GPU (T4) · Wav2Lip 口型同步**

## 操作步骤
1. 先点击菜单：`运行时 → 更改运行时类型 → GPU (T4)`
2. 上传你的保险客服正面照片（左侧文件图标 → 上传 → 命名 `portrait.jpg`）
3. 依次运行每个代码块（点播放按钮 ▶）
4. 最后自动下载 5 个 .webm 视频文件
5. 把 .webm 放到项目中即可使用

In [ ]:
# 1. 安装依赖 (约2分钟)
!pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python librosa numpy==1.23.5 tqdm scipy face-alignment
!pip install -q edge-tts
!apt-get update -qq && apt-get install -y -qq ffmpeg
print('✅ 依赖安装完成')

In [ ]:
# 2. 克隆 Wav2Lip + 下载模型 (约3分钟)
import os
if not os.path.exists('Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip

os.makedirs('Wav2Lip/checkpoints', exist_ok=True)
os.makedirs('Wav2Lip/face_detection/detection/sfd', exist_ok=True)

# 下载 Wav2Lip GAN 模型
!gdown --id 1fQtBSYEyuai9MjBOF8j1zZ6o7bAewR-Q -O Wav2Lip/checkpoints/wav2lip_gan.pth 2>/dev/null

# 下载人脸检测模型
!gdown --id 1EZpsEdBkxD3jY6r2j8Z5uF6D7Qx7bHnQ -O Wav2Lip/face_detection/detection/sfd/s3fd.pth 2>/dev/null

print('✅ Wav2Lip 准备完成')

In [ ]:
# 3. 边缘TTS生成语音 (约30秒)
import asyncio, edge_tts, os

ANSWERS = {
  'greeting': '您好！我是智能保险客服，欢迎您。请问投保流程、理赔流程、退保规则、续保政策，你想了解哪方面？',
  'claim': '理赔流程如下：第一步，拨打95511报案。第二步，准备身份证、保单、医疗记录。第三步，提交审核，三到五个工作日内完成。第四步，赔付金额到账。',
  'insurance': '投保流程如下：第一步，选择产品，包括寿险、健康险、意外险、车险。第二步，填写个人信息。第三步，健康告知。第四步，支付保费。第五步，保单生效。二十四小时在线投保。',
  'refund': '退保规则：投保后的十天犹豫期内，可以全额退款。超过犹豫期后，按照现金价值进行返还。办理退保需要准备保单、身份证和银行卡。',
  'renewal': '续保政策：保障到期前的三十天内，可以办理续保手续。续保不需要重新核保，按照原费率或者调整后的费率计算即可。'
}

os.makedirs('/content/audio', exist_ok=True)

async def gen_tts():
  for name, text in ANSWERS.items():
    path = f'/content/audio/{name}.wav'
    if os.path.exists(path): continue
    print(f'  {name}...', end=' ')
    comm = edge_tts.Communicate(text, 'zh-CN-XiaoxiaoNeural')
    await comm.save(path)
    print('✓')
  print('✅ 语音生成完成')

await gen_tts()

In [ ]:
# 4. 检查照片是否已上传
import os, IPython.display as ipd
if os.path.exists('/content/portrait.jpg'):
    print('✅ 照片已上传')
else:
    print('❌ 请先上传照片！')
    print('在左侧文件图标 → 上传 → 命名为 portrait.jpg')
    print('照片要求：正面、半身、蓝色西装最佳')

In [ ]:
# 5. Wav2Lip 生成真人级口型同步视频 (约5分钟)
import subprocess, os
os.makedirs('/content/output', exist_ok=True)

MAP = {
  'greeting': 'greeting',
  'claim': 'claim-guide',
  'insurance': 'insurance-intro',
  'refund': 'refund-info',
  'renewal': 'renewal-info'
}

for key, outname in MAP.items():
  print(f'生成 {outname}...')
  cmd = [
    'python', 'Wav2Lip/inference.py',
    '--checkpoint_path', 'Wav2Lip/checkpoints/wav2lip_gan.pth',
    '--face', '/content/portrait.jpg',
    '--audio', f'/content/audio/{key}.wav',
    '--outfile', f'/content/output/{outname}_temp.mp4',
    '--pads', '0', '20', '0', '0',
    '--resize_factor', '1'
  ]
  subprocess.run(cmd, check=True)
  
  # 转 WebM
  subprocess.run([
    'ffmpeg', '-y', '-i', f'/content/output/{outname}_temp.mp4',
    '-c:v', 'libvpx-vp9', '-b:v', '500k', '-crf', '30',
    '-vf', 'scale=640:480', '-an', '-shortest',
    f'/content/output/{outname}.webm'
  ], check=True)
  print(f'  ✅ {outname}.webm 完成')

print('\n🎉 全部生成完成！')

In [ ]:
# 6. 下载所有视频文件
from google.colab import files
import os

videos = ['greeting','claim-guide','insurance-intro','refund-info','renewal-info']
for v in videos:
    path = f'/content/output/{v}.webm'
    if os.path.exists(path):
        print(f'下载 {v}.webm ({os.path.getsize(path)//1024}KB)...')
        files.download(path)

print('\n📁 下载完成后，把这5个文件放到项目的 web/public/avatar-videos/ 目录')
print('然后运行项目即可看到真人级数字人客服！')